# 🏪 FahMai RAG System — Full Production Pipeline

**SuperAI Engineer Season 6 — Level 1 Hackathon**

This notebook builds a high-accuracy RAG system for FahMai electronics store Q&A. ( Fahsai RAG System)

## Pipeline Features
**Smart Chunking**: ใช้การตัดแบ่งข้อความโดยคำนึงถึงโครงสร้าง (หัวข้อ/ส่วนต่างๆ) และมีการซ้อนทับกัน (Overlap)

**Contextual Chunks**: ข้อความแต่ละส่วนจะถูกเสริมด้วยข้อมูลอภิพันธุ์ (Metadata) ของเอกสารนั้นๆ เพื่อให้ระบบสามารถค้นคืนได้ดียิ่งขึ้น

 **Hybrid Retrieval**: ใช้การค้นหาทั้งแบบ Dense (เวกเตอร์ฝังตัวที่รองรับหลายภาษา) และแบบ Sparse (BM25) จากนั้นนำมารวมผลลัพธ์เข้าด้วยกันด้วยเทคนิค RRF Fusion

**Cosine Similarity**: ใช้เวกเตอร์ที่ผ่านการทำ Normalization เพื่อให้การให้คะแนนมีความแม่นยำ

**Thai ThaiLLM**: ใช้โมเดล openthaigpt สำหรับการสร้างคำตอบ

**Smart Prompt Engineering**: การใช้เทคนิค Few-shot พร้อมกับการออกแบบโครงสร้างพรอมต์ภาษาไทยอย่างเป็นระบบ

**Eval Analysis**: มีระบบการประเมินผลตัวเองและการให้คะแนนความมั่นใจ (Confidence scoring) ในคำตอบ
**Submission CSV**: สามารถสร้างไฟล์ผลลัพธ์คำตอบออกมาได้โดยอัตโนมัติ

## Sections
0. Setup & Configuration
1. Load & Process Knowledge Base
2. Smart Chunking + Contextual Enrichment
3. Embedding (Multilingual Dense)
4. BM25 Sparse Index
5. Hybrid Retrieval (RRF)
6. ThaiLLM Generation
7. Run Full Pipeline
8. Eval Analysis
9. Generate Submission

##*Accuracy  = public score 0.78 to private score : 0.85 *

## Section 0: Setup & Configuration

In [ ]:
# Install dependencies
!pip install -q sentence-transformers pythainlp rank-bm25 requests python-dotenv numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 63.6 MB/s eta 0:00:00


In [ ]:
!unzip super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

Archive:  super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip
  inflating: data/knowledge_base/policies/cancellation_policy.md  
  inflating: data/knowledge_base/policies/membership_points_policy.md  
  inflating: data/knowledge_base/policies/return_policy.md  
  inflating: data/knowledge_base/policies/shipping_policy.md  
  inflating: data/knowledge_base/policies/warranty_policy.md  
  inflating: data/knowledge_base/products/AW-MN-001_arcwave_proview_27_4k.md  
  inflating: data/knowledge_base/products/AW-SK-001_arcwave_soundpillar_300.md  
  inflating: data/knowledge_base/products/DN-DT-001_daonuea_tower_x10.md  
  inflating: data/knowledge_base/products/DN-DT-002_daonuea_tower_x10_max.md  
  inflating: data/knowledge_base/products/DN-DT-003_daonuea_mini_pc_m1.md  
  inflating: data/knowledge_base/products/DN-DT-004_daonuea_all_in_one_27.md  
  inflating: data/knowledge_base/products/DN-DT-005_daonuea_all_in_one_24.md  
  inflating: data/knowledge_base/products/DN-LT-001_daonuea_

In [ ]:
# CONFIGURATION
import os

# PATH CONFIG
DATA_DIR = "/content/data"

KB_DIR = f"{DATA_DIR}/knowledge_base"
QUESTIONS_CSV = f"{DATA_DIR}/questions.csv"
SAMPLE_SUB_CSV = f"{DATA_DIR}/sample_submission.csv"
OUTPUT_CSV = "submission.csv"
CHECKPOINT_FILE = "checkpoint_predictions.json"

# Run Config
N_QUESTIONS = 100
TOP_K = 10
CHUNK_SIZE = 350
CHUNK_OVERLAP = 80

# Model Config
EMBED_MODEL_NAME = "BAAI/bge-m3"

 # API KEY ThaiLLM
THAILLM_API_KEY = ""(อยุ่ใน secret key)
#  LLM MODEL
LLM_MODEL = "openthaigpt"

#  แสดงผล
masked = THAILLM_API_KEY[:4] + "***" + THAILLM_API_KEY[-4:]
print(f"✅ API Key: {masked}")
print(f"🤖 LLM model: {LLM_MODEL}")
print(f"🧮 Embedding: {EMBED_MODEL_NAME}")
print(f"🔍 Top-K: {TOP_K}  |  Chunk size: {CHUNK_SIZE}  |  Questions: {N_QUESTIONS}")


✅ API Key: ds08***zKt5
🤖 LLM model: openthaigpt
🧮 Embedding: BAAI/bge-m3
🔍 Top-K: 10  |  Chunk size: 350  |  Questions: 100


In [ ]:
# ทดสอบการตรวจสอบความถูกต้องของ API KEY
# แก้ไข: ลบฟังก์ชัน ask_llm() ตัวเก่าที่ใช้งานไม่ได้ออกจากส่วนนี้ (เนื่องจากไม่มีระบบ Retry / การจัดการ Error)
# ฟังก์ชัน ask_llm() ตัวที่ถูกต้องพร้อมระบบ Retry เต็มรูปแบบ ถูกกำหนดไว้ในส่วนของ Helper Functions ด้านล่าง
# ปัจจุบันเซลล์ (Cell) นี้มีหน้าที่ตรวจสอบความถูกต้องของ API key และ URL ก่อนที่จะรันคำสั่งใดๆ

import requests

def _test_api_key(api_key: str, model: str = "typhoon") -> bool:
    """Quick smoke test to verify API key is valid before running full pipeline."""
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": api_key}
    payload = {
        "model": "/model",
        "messages": [{"role": "user", "content": "สวัสดี ตอบสั้นๆ"}],
        "max_tokens": 20,
        "temperature": 0.3,
    }
    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        if resp.status_code == 401:
            print(f"❌ API Key INVALID (401 Unauthorized) — check your ThaiLLM secret")
            return False
        if resp.status_code == 404:
            print(f"❌ Model not found (404) — model '{model}' may not exist")
            return False
        if resp.status_code == 200:
            data = resp.json()
            reply = data["choices"][0]["message"]["content"].strip()
            print(f"✅ API Key VALID — model={model} responded: '{reply[:50]}'")
            return True
        print(f"⚠️  Unexpected status {resp.status_code}: {resp.text[:200]}")
        return False
    except requests.exceptions.ConnectionError:
        print("❌ Cannot reach thaillm.or.th — check network connectivity")
        return False
    except requests.exceptions.Timeout:
        print("⚠️  API test timed out — server may be slow, but key might still work")
        return True
    except Exception as e:
        print(f"⚠️  Test error: {e}")
        return False

print("🔍 Testing ThaiLLM API connection...")
_api_ok = _test_api_key(THAILLM_API_KEY, LLM_MODEL)
if not _api_ok:
    print("\n⚠️  Fix API key before running the pipeline or you will get 0% accuracy!")
else:
    print("   Ready to run pipeline.")


🔍 Testing ThaiLLM API connection...
✅ API Key VALID — model=openthaigpt responded: '<think>
ผู้ใช้ต้องการให้ตอบคำถามด้วยภาษาไทยเป็น'
   Ready to run pipeline.


## Section 1: Helper Functions & Load Data

In [ ]:
import os, csv, re, time, json, math, requests, random
import numpy as np
from pathlib import Path
from collections import defaultdict

# HELPER FUNCTIONS

def normalize_text(text: str) -> str:
    """Normalize Thai/English text: collapse whitespace, strip."""
    text = re.sub(r'\r\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    if a.ndim == 1:
        norm_a = np.linalg.norm(a)
        norm_b = np.linalg.norm(b)
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return float(np.dot(a, b) / (norm_a * norm_b))
    else:
        norm_a = np.linalg.norm(a, axis=1, keepdims=True)
        norm_b = np.linalg.norm(b)
        if norm_b == 0:
            return np.zeros(len(a))
        safe_norm_a = np.where(norm_a == 0, 1, norm_a)
        return (a / safe_norm_a) @ (b / norm_b)


def extract_doc_metadata(text: str, filepath: str) -> dict:
    """Extract metadata from a markdown knowledge base document."""
    meta = {"filepath": filepath, "product_code": "", "product_name": "",
            "brand": "", "category": "", "price": ""}

    fname = Path(filepath).stem
    code_match = re.match(r'^([A-Z]{2}-[A-Z]{2}-\d+|[A-Z]+-[A-Z]+-\d+)', fname.upper())
    if code_match:
        meta["product_code"] = code_match.group(1)

    lines = text.split('\n')
    for line in lines[:20]:
        line = line.strip()
        if line.startswith('# '):
            meta["product_name"] = line[2:].strip()
        if 'รหัสสินค้า:' in line or 'รหัส:' in line:
            meta["product_code"] = line.split(':', 1)[-1].strip()
        if 'แบรนด์:' in line:
            meta["brand"] = line.split(':', 1)[-1].strip()
        if 'หมวดหมู่:' in line:
            meta["category"] = line.split(':', 1)[-1].strip()
        if 'ราคา:' in line:
            meta["price"] = line.split(':', 1)[-1].strip()

    return meta


def ask_llm(messages, model=None, max_retries=6, base_delay=1.0):

    if model is None:
        model = LLM_MODEL  # Read global at CALL time, not define time

    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2048,
        "temperature": 0.3,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=45)

            # FIX: Fast-fail on auth/not-found errors (no point retrying)
            if resp.status_code == 401:
                print(f"  ❌ 401 Unauthorized — check THAILLM_API_KEY")
                return None

            if resp.status_code == 404:
                print(f"  ❌ 404 Model not found: {model}")
                return None

            # Rate limited — exponential backoff + jitter
            if resp.status_code == 429:
                wait = min(base_delay * (2 ** attempt), 60) + random.uniform(0, 2)
                print(f"  ⏳ Rate limited (429), waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue

            # Service unavailable
            if resp.status_code in (502, 503, 504):
                wait = min(base_delay * (2 ** attempt), 30) + random.uniform(0, 1)
                print(f"  ⚠️  {resp.status_code} server error, waiting {wait:.1f}s")
                time.sleep(wait)
                continue

            resp.raise_for_status()

            # FIX: Extract text content (not raw dict)
            data = resp.json()
            content = data["choices"][0]["message"]["content"].strip()
            return content

        except requests.exceptions.Timeout:
            wait = min(base_delay * (2 ** attempt), 30) + random.uniform(0, 2)
            print(f"  ⏰ Timeout (45s), retrying in {wait:.1f}s (attempt {attempt+1})")
            time.sleep(wait)

        except requests.exceptions.ConnectionError as e:
            wait = min(base_delay * (2 ** attempt), 30)
            print(f"  🔌 Connection error, retrying in {wait:.1f}s: {e}")
            time.sleep(wait)

        except (KeyError, IndexError, json.JSONDecodeError) as e:
            # Response parse error — log and retry once
            print(f"  📋 Response parse error: {e} | resp: {getattr(resp, 'text', '')[:200]}")
            if attempt < max_retries - 1:
                time.sleep(2)
            continue

        except requests.exceptions.RequestException as e:
            wait = min(base_delay * (2 ** attempt), 30)
            print(f"  ❌ Request error: {e}, retrying in {wait:.1f}s")
            time.sleep(wait)

    print(f" All {max_retries} retries exhausted for this question")
    return None


def parse_answer(text: str):
    """Extract integer answer 1-10 from LLM response.
    Handles: 'ANSWER: 5', 'ตอบ: 3', standalone digits, chain-of-thought.
    """
    if text is None:
        return None

    # Remove <think>...</think> chain-of-thought blocks
    clean = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

    # Priority 1: ANSWER: X pattern (most reliable)
    m = re.search(r'ANSWER[:\s]+([1-9]|10)\b', clean, re.IGNORECASE)
    if m:
        return int(m.group(1))

    # Priority 2: Thai answer patterns
    m = re.search(r'(?:ตอบ|คำตอบ|เลือก)[:\s]+([1-9]|10)\b', clean)
    if m:
        return int(m.group(1))

    # Priority 3: Last standalone number in response
    nums = re.findall(r'\b([1-9]|10)\b', clean)
    if nums:
        return int(nums[-1])

    return None


def load_questions(path: str) -> list:
    """Load questions CSV into structured list."""
    questions = []
    with open(path, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            choices = {str(i): row[f'choice_{i}'] for i in range(1, 11)}
            questions.append({
                'id': int(row['id']),
                'question': row['question'],
                'choices': choices
            })
    return questions


def write_submission(predictions: dict, questions: list, path: str):
    """Write submission CSV file."""
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'answer'])
        for q in questions:
            qid = q['id']
            ans = predictions.get(qid, 9)
            writer.writerow([qid, ans])
    print(f"✅ Submission written to {path} ({len(questions)} rows)")


def save_checkpoint(predictions: dict, raw_responses: dict, path: str = CHECKPOINT_FILE):
    """Save current predictions to disk for crash recovery."""
    data = {"predictions": {str(k): v for k, v in predictions.items()},
            "raw_responses": {str(k): v for k, v in raw_responses.items()}}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)


def load_checkpoint(path: str = CHECKPOINT_FILE) -> tuple:
    """Load previously saved predictions (crash recovery)."""
    if not os.path.exists(path):
        return {}, {}
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    preds = {int(k): v for k, v in data.get("predictions", {}).items()}
    raws = {int(k): v for k, v in data.get("raw_responses", {}).items()}
    print(f"♻️  Loaded checkpoint: {len(preds)} existing predictions")
    return preds, raws


print("✅ Helper functions loaded (with all bug fixes)")


✅ Helper functions loaded (with all bug fixes)


In [ ]:
# Load all knowledge base documents
kb_dir = Path(KB_DIR)
documents = []

for fp in sorted(kb_dir.rglob('*.md')):
    raw_text = fp.read_text(encoding='utf-8')
    text = normalize_text(raw_text)
    rel_path = str(fp.relative_to(kb_dir))
    meta = extract_doc_metadata(text, rel_path)

    # Determine doc type
    if 'products/' in rel_path:
        doc_type = 'product'
    elif 'policies/' in rel_path:
        doc_type = 'policy'
    else:
        doc_type = 'store_info'

    documents.append({
        'path': rel_path,
        'text': text,
        'meta': meta,
        'doc_type': doc_type,
    })

print(f"📚 Loaded {len(documents)} documents")
print(f"   products:   {sum(1 for d in documents if d['doc_type'] == 'product')}")
print(f"   policies:   {sum(1 for d in documents if d['doc_type'] == 'policy')}")
print(f"   store_info: {sum(1 for d in documents if d['doc_type'] == 'store_info')}")

# Load questions
questions = load_questions(QUESTIONS_CSV)
print(f"\n❓ Loaded {len(questions)} questions")
print(f"   Example Q1: {questions[0]['question']}")

📚 Loaded 118 documents
   products:   110
   policies:   5
   store_info: 3

❓ Loaded 100 questions
   Example Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ


## Section 2: Smart Chunking + Contextual Enrichment

In [ ]:
# SMART CHUNKING STRATEGY

# กลยุทธ์: การตัดแบ่งข้อความโดยคำนึงถึงโครงสร้าง (Structure-aware chunking)
# 1. พยายามตัดแบ่งตามหัวข้อ Markdown ก่อน (เช่น ## ชื่อส่วนต่างๆ)
# 2. หากไม่มีหัวข้อ ให้เปลี่ยนไปตัดแบ่งตามแถวของตาราง หรือตามบล็อกของย่อหน้าแทน
# 3. เพิ่มข้อความระบุบริบท (Contextual prefix) ไว้ด้านหน้าของทุกก้อนข้อความ เช่น: ชื่อเอกสาร + หมวดหมู่ + ข้อมูลสินค้า
# วิธีนี้จะช่วยรับประกันว่า ข้อความแต่ละก้อน (Chunk) จะมีใจความสมบูรณ์ในตัวเอง และพร้อมสำหรับการถูกค้นคืน (Retrieval)

def split_by_sections(text: str) -> list:
    """Split markdown doc by ## headers into sections."""
    # Split on level-2 headers (## ...) while keeping the header
    parts = re.split(r'(?=^## )', text, flags=re.MULTILINE)
    return [p.strip() for p in parts if p.strip()]


def fixed_size_chunks(text: str, size: int, overlap: int) -> list:
    """Fixed-size sliding window chunking with overlap."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += size - overlap
    return chunks


def make_contextual_prefix(doc: dict) -> str:
    meta = doc['meta']
    parts = []

    if doc['doc_type'] == 'product':
        if meta['product_name']:
            parts.append(f"สินค้า: {meta['product_name']}")
        if meta['product_code']:
            parts.append(f"รหัส: {meta['product_code']}")
        if meta['brand']:
            parts.append(f"แบรนด์: {meta['brand']}")
        if meta['category']:
            parts.append(f"หมวดหมู่: {meta['category']}")
        if meta['price']:
            parts.append(f"ราคา: {meta['price']}")
    elif doc['doc_type'] == 'policy':
        fname = Path(doc['path']).stem
        parts.append(f"นโยบาย: {meta['product_name'] or fname}")
        parts.append("ประเภท: นโยบายร้านฟ้าใหม่")
    else:
        fname = Path(doc['path']).stem
        parts.append(f"ข้อมูลร้าน: {meta['product_name'] or fname}")
        parts.append("ประเภท: ข้อมูลทั่วไปร้านฟ้าใหม่")

    return ' | '.join(parts)


def build_chunks(documents: list, chunk_size: int = CHUNK_SIZE,
                  overlap: int = CHUNK_OVERLAP) -> list:
    """Build enriched chunks from all documents."""
    all_chunks = []

    for doc in documents:
        prefix = make_contextual_prefix(doc)

        # Strategy: try section-based split first
        sections = split_by_sections(doc['text'])

        for section in sections:
            if len(section) <= chunk_size:
                # Section fits in one chunk — add with prefix
                chunk_text = f"{prefix}\n\n{section}" if prefix else section
                all_chunks.append({
                    'text': chunk_text,
                    'raw_text': section,  # Original for BM25
                    'source': doc['path'],
                    'doc_type': doc['doc_type'],
                    'meta': doc['meta'],
                })
            else:
                # Section too large — further split with sliding window
                sub_chunks = fixed_size_chunks(section, chunk_size, overlap)
                for sc in sub_chunks:
                    chunk_text = f"{prefix}\n\n{sc}" if prefix else sc
                    all_chunks.append({
                        'text': chunk_text,
                        'raw_text': sc,
                        'source': doc['path'],
                        'doc_type': doc['doc_type'],
                        'meta': doc['meta'],
                    })

    return all_chunks


chunks = build_chunks(documents)
print(f"✅ Created {len(chunks)} chunks from {len(documents)} documents")
print(f"   Avg chunk size: {np.mean([len(c['text']) for c in chunks]):.0f} chars")
print(f"   Max chunk size: {max(len(c['text']) for c in chunks)} chars")

# Show sample
print(f"\n--- Sample Chunk (first product) ---")
sample = next(c for c in chunks if c['doc_type'] == 'product')
print(f"Source: {sample['source']}")
print(sample['text'][:400], "...")

✅ Created 1680 chunks from 118 documents
   Avg chunk size: 425 chars
   Max chunk size: 589 chars

--- Sample Chunk (first product) ---
Source: products/AW-MN-001_arcwave_proview_27_4k.md
สินค้า: อาร์คเวฟ ProView 27 4K (ArcWave ProView 27 4K) | รหัส: AW-MN-001 | แบรนด์: อาร์คเวฟ (ArcWave) — แบรนด์พันธมิตร | หมวดหมู่: จอมอนิเตอร์ | ราคา: ฿12,990

# อาร์คเวฟ ProView 27 4K (ArcWave ProView 27 4K)

รหัสสินค้า: AW-MN-001
แบรนด์: อาร์คเวฟ (ArcWave) — แบรนด์พันธมิตร
หมวดหมู่: จอมอนิเตอร์
ราคา: ฿12,990
สถานะ: มีสินค้า
วันที่อัปเดต: 1 มีนาคม 2569 ...


## Section 3: Dense Embedding (Multilingual)

In [ ]:
from sentence_transformers import SentenceTransformer

def embed(texts):
    return embed_model.encode(
        texts,
        normalize_embeddings=True
    )

print(f"🔄 Loading embedding model: {EMBED_MODEL_NAME}")
print("   (Downloads ~400MB on first run, cached afterwards)")

embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"✅ Model loaded: {EMBED_MODEL_NAME}")
print(f"   Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

# Embed all chunks
print(f"\n🔄 Embedding {len(chunks)} chunks...")
chunk_texts = [c['text'] for c in chunks]
chunk_embeddings = embed_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True  # L2 normalize → cosine sim = dot product
)

print(f"✅ Chunk embeddings: {chunk_embeddings.shape}")
print(f"   Shape: (n_chunks={chunk_embeddings.shape[0]}, dim={chunk_embeddings.shape[1]})")

# Verify normalization (should be ~1.0)
norms = np.linalg.norm(chunk_embeddings[:5], axis=1)
print(f"   Norm check (should be ~1.0): {norms}")

🔄 Loading embedding model: BAAI/bge-m3
   (Downloads ~400MB on first run, cached afterwards)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Model loaded: BAAI/bge-m3
   Embedding dimension: 1024

🔄 Embedding 1680 chunks...


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Server error '504 Gateway Time-out' for url 'https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversi

✅ Chunk embeddings: (1680, 1024)
   Shape: (n_chunks=1680, dim=1024)
   Norm check (should be ~1.0): [1. 1. 1. 1. 1.]


## Section 4: BM25 Sparse Index

In [ ]:
from rank_bm25 import BM25Okapi
from pythainlp.tokenize import word_tokenize

# BM25 SPARSE RETRIEVAL

# จำเป็นต้องมีการตัดคำภาษาไทยก่อนที่จะนำไปใช้กับอัลกอริทึม BM25
# ใช้ระบบตัดคำ (engine) ชื่อ 'newmm' ของไลบรารี pythainlp: ซึ่งมีความสมดุลที่ดีเยี่ยมระหว่างความเร็วและความแม่นยำ
# เราจะใช้ raw_text (ข้อความดั้งเดิมที่ยังไม่ได้เติมคำนำหน้าหรือ Contextual prefix) สำหรับการทำ BM25
# เพื่อป้องกันไม่ให้คำในส่วนคำนำหน้า มีอิทธิพลหรือได้คะแนนสูงเกินไปจนกลบการจับคู่คำสำคัญ (Keyword) ของเนื้อหาจริงๆ

def thai_tokenize(text: str) -> list:
    """Tokenize Thai text, keeping numbers and English words."""
    tokens = word_tokenize(text, engine='newmm', keep_whitespace=False)
    # Filter very short tokens and whitespace
    tokens = [t.strip() for t in tokens if t.strip() and len(t.strip()) > 0]
    return tokens


print("🔄 Tokenizing chunks for BM25...")

# Use full enriched text for BM25 (includes metadata prefix)
# This lets BM25 match product names, codes, etc.
tokenized_chunks = [thai_tokenize(c['text']) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"✅ BM25 index built over {len(tokenized_chunks)} chunks")

# Demo tokenization
sample_q = "Watch S3 Ultra กันน้ำได้กี่ ATM"
print(f"\nDemo tokenization:")
print(f"  Input:  '{sample_q}'")
print(f"  Tokens: {thai_tokenize(sample_q)}")

🔄 Tokenizing chunks for BM25...
✅ BM25 index built over 1680 chunks

Demo tokenization:
  Input:  'Watch S3 Ultra กันน้ำได้กี่ ATM'
  Tokens: ['Watch', 'S', '3', 'Ultra', 'กันน้ำ', 'ได้', 'กี่', 'ATM']


## Section 5: Hybrid Retrieval (RRF)

In [ ]:
# RETRIEVAL FUNCTIONS


def dense_retrieve(query: str, k: int = TOP_K) -> list:
    """Dense semantic retrieval using cosine similarity.
    Returns list of (chunk_idx, similarity_score) tuples.
    """
    q_emb = embed_model.encode([query], normalize_embeddings=True)[0]

    # Cosine similarity: since both are normalized, this = dot product
    # We also have our custom cosine_similarity() for verification
    scores = np.dot(chunk_embeddings, q_emb)

    top_idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_idx]


def bm25_retrieve(query: str, k: int = TOP_K) -> list:
    """BM25 sparse keyword retrieval.
    Returns list of (chunk_idx, bm25_score) tuples.
    """
    tokens = thai_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_idx]


def hybrid_retrieve(query: str, k: int = TOP_K, rrf_k: int = 60,
                     fetch_mult: int = 3) -> list:

    fetch_k = k * fetch_mult

    dense_results = dense_retrieve(query, k=fetch_k)
    bm25_results = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = defaultdict(float)

    for rank, (idx, _) in enumerate(dense_results, 1):
        rrf_scores[idx] += 1.0 / (rrf_k + rank)

    for rank, (idx, _) in enumerate(bm25_results, 1):
        rrf_scores[idx] += 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return sorted_results


def get_chunks_by_results(results: list) -> list:
    """Convert (idx, score) list to list of chunk dicts."""
    return [chunks[idx] for idx, _ in results]

# DEMO: Compare retrieval methods
test_q = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ"
print(f"Query: {test_q}\n")

d_res = dense_retrieve(test_q, k=5)
b_res = bm25_retrieve(test_q, k=5)
h_res = hybrid_retrieve(test_q, k=5)

print("Dense top-5:")
for idx, score in d_res:
    print(f"  [{score:.3f}] {chunks[idx]['source']}")

print("\nBM25 top-5:")
for idx, score in b_res:
    print(f"  [{score:.3f}] {chunks[idx]['source']}")

print("\nHybrid (RRF) top-5:")
for idx, score in h_res:
    print(f"  [{score:.4f}] {chunks[idx]['source']}")

Query: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Dense top-5:
  [0.711] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [0.692] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [0.661] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [0.657] products/WK-SW-002_wongkhojon_watch_s3_pro.md
  [0.644] products/WK-SW-001_wongkhojon_watch_s3_ultra.md

BM25 top-5:
  [24.498] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [23.202] products/WK-SW-002_wongkhojon_watch_s3_pro.md
  [22.213] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [21.434] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [20.084] products/WK-SW-003_wongkhojon_watch_s3.md

Hybrid (RRF) top-5:
  [0.0323] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [0.0323] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [0.0318] products/WK-SW-002_wongkhojon_watch_s3_pro.md
  [0.0308] products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  [0.0303] products/WK-SW-001_wongkhojon_watch_s3_ultra.md


## Section 6: ThaiLLM Generation + Prompt Engineering

In [ ]:
# SYSTEM PROMPT — Optimized for high accuracy Thai MC
# 1. Explicit "read carefully" + step-by-step reasoning instruction
# 2. Specific handling for numerical/spec questions
# 3. Clear fallback rules (choice 9 vs 10)

SYSTEM_PROMPT = """คุณคือผู้เชี่ยวชาญด้านสินค้าอิเล็กทรอนิกส์ของร้านฟ้าใหม่ (FahMai)

วิธีตอบคำถามปรนัย 10 ตัวเลือก:
1. อ่านข้อมูลอ้างอิงทั้งหมดอย่างละเอียด
2. หาข้อมูลที่ตรงกับคำถามโดยตรง (ชื่อสินค้า, รหัส, สเปค, ราคา, นโยบาย)
3. จับคู่คำตอบที่ถูกต้องกับตัวเลือก 1-10
4. ถ้าไม่มีข้อมูลในเอกสาร → ตอบ 9
5. ถ้าคำถามไม่เกี่ยวกับร้านฟ้าใหม่เลย → ตอบ 10

กฎสำคัญ:
- ตอบเฉพาะ ANSWER: X เท่านั้น (X = เลข 1-10)
- ห้ามเดาถ้าไม่มีข้อมูลในเอกสาร → ตอบ 9
- สำหรับตัวเลข/สเปค: ต้องตรงทุกหน่วย (ATM, mAh, นิ้ว, บาท)"""


def build_rag_prompt(question: str, choices: dict, retrieved_chunks: list) -> str:
    """Build RAG prompt: context + question + choices.
    IMPROVEMENT: Numbered context sources for LLM to reference specifically.
    """
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        src = Path(chunk['source']).stem
        context_parts.append(f"[เอกสาร {i}: {src}]\n{chunk['text']}")
    context = "\n\n---\n\n".join(context_parts)

    choices_text = "\n".join(f"  {k}. {v}" for k, v in choices.items())

    return f"""== ข้อมูลอ้างอิงจากร้านฟ้าใหม่ ==
{context}

== คำถาม ==
{question}

== ตัวเลือก ==
{choices_text}

วิเคราะห์จากข้อมูลอ้างอิงและตอบ: ANSWER: X"""


# DEMO: Single question
q = questions[0]
h_res = hybrid_retrieve(q['question'])
retrieved = get_chunks_by_results(h_res)

prompt = build_rag_prompt(q['question'], q['choices'], retrieved)
print(f"Q{q['id']}: {q['question']}")
print(f"\nPrompt preview (first 800 chars):")
print(prompt[:800])
print("...")


Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Prompt preview (first 800 chars):
== ข้อมูลอ้างอิงจากร้านฟ้าใหม่ ==
[เอกสาร 1: WK-SW-001_wongkhojon_watch_s3_ultra]
สินค้า: วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) | รหัส: WK-SW-001 | แบรนด์: วงโคจร (WongKhoJon) — แบรนด์ในเครือฟ้าใหม่ | หมวดหมู่: สมาร์ทวอทช์ | ราคา: ฿14,990

มใส่ที่เบาสบายตลอดทั้งวันแม้ระหว่างออกกำลังกายหนัก

ความโดดเด่นที่สุดของ S3 Ultra คือมาตรฐานกันน้ำระดับ 100 เมตร (10 ATM) พร้อมโหมด Dive Mode ที่รองรับการดำน้ำได้ถึง 40 เมตร บันทึกความลึก ความดัน อุณหภูมิน้ำ และเวลาดำน้ำแบบ real-time แสดงผลบนหน้าจอ AMOLED 1.9 นิ้วที่ยังคงอ่านได้ชัดเจนใต้น้ำ รองรับการดำน้ำทั้งแบบ recreational และ freediving

ระบบ GP

---

[เอกสาร 2: WK-SW-001_wongkhojon_watch_s3_ultra]
สินค้า: วงโคจร Watch S3 Ultra (WongKhoJon Watch S3 Ultra) | รหัส: WK-SW-001 | แบรนด์: วงโคจร (WongKhoJon) — แบรนด์ในเครือฟ้าใหม่ | หมวดหมู่: สม
...


In [ ]:
# Test the full RAG pipeline on Q1
q = questions[0]
print(f"Testing on: Q{q['id']}: {q['question']}")
print(f"Choices:")
for k, v in q['choices'].items():
    print(f"  {k}. {v}")

h_res = hybrid_retrieve(q['question'])
retrieved = get_chunks_by_results(h_res)
prompt = build_rag_prompt(q['question'], q['choices'], retrieved)

raw = ask_llm([
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': prompt},
])

answer = parse_answer(raw)
print(f"\nLLM raw response: {raw}")
print(f"Parsed answer: {answer}")
if answer:
    print(f"Answer text: {q['choices'][str(answer)]}")

Testing on: Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
Choices:
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า

LLM raw response: <think>
คำถามคือ "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ" ต้องหาข้อมูลจากเอกสารอ้างอิงที่ให้มา

1.  **เอกสาร 1:** ระบุว่า "มาตรฐานกันน้ำระดับ 100 เมตร (10 ATM)"
2.  **เอกสาร 5:** ระบุว่า "กันน้ำ | 100 เมตร (10 ATM) + Dive Mode (สูงสุด 40 เมตร)"
3.  **เอกสาร 6:** ระบุว่า "Dive Mode ของ S3 Ultra รองรับการดำน้ำสูงสุด 40 เมตร (ลึกกว่านั้นนาฬิกาจะไม่รับประกันความถูกต้องของข้อมูลและอาจเกิดความเสียหายได้)" ซึ่งยืนยันว่ากันน้ำได้ 10 ATM

ดังนั้น คำตอบที่ถูกต้องคือ 10 ATM ซึ่งตรงกับตัวเลือกที่ 5
</think>

ANSWER: 5
Parsed answer: 5
Answer text: 10 ATM


## Section 7: Full Pipeline Run

In [ ]:
# MAIN PIPELINE RUNNER
# 1. บันทึกจุดตรวจสอบ (Checkpoint) ทุกๆ 10 คำถาม (เพื่อกู้คืนข้อมูลหากโปรแกรมหรือระบบล่ม)
# 2. สามารถรันทำงานต่อจาก Checkpoint เดิมได้ทันที หากมีไฟล์เซฟไว้ (ไม่ต้องเริ่มรันข้อ 1 ใหม่)
# 3. หน่วงเวลาแบบปรับได้ (Adaptive delay): ใช้เวลาพื้นฐานที่ 1.0 วินาที (จากเดิม 0.3 วินาที) เพื่อป้องกันการถูกจำกัดการใช้งาน API (Rate limits)
# 4. ปรับปรุงหน้าจอแสดงผลความคืบหน้า (Progress display) ให้ดีขึ้น พร้อมระบบติดตามอัตราความสำเร็จ (Success rate tracking)

def run_rag_pipeline(questions: list, n: int = N_QUESTIONS,
                      model: str = None,
                      delay: float = 1.0,
                      resume_from_checkpoint: bool = True) -> tuple:
    if model is None:
        model = LLM_MODEL

    # FIX: Resume from checkpoint (crash recovery)
    if resume_from_checkpoint:
        predictions, raw_responses = load_checkpoint()
    else:
        predictions, raw_responses = {}, {}

    target_qs = questions[:n]
    # Skip already answered
    remaining = [q for q in target_qs if q['id'] not in predictions]

    print(f"🚀 Running RAG on {len(target_qs)} questions (model={model})")
    print(f"   Already done: {len(predictions)} | Remaining: {len(remaining)}")
    print(f"   Retrieval: Hybrid (Dense + BM25 RRF), top_k={TOP_K}")
    print(f"   Delay between calls: {delay}s")
    print(f"   {'='*50}")

    if not remaining:
        print("✅ All questions already answered (checkpoint). Nothing to do.")
        return predictions, raw_responses

    start_time = time.time()
    success_count = 0
    fallback_count = 0

    for i, q in enumerate(remaining, 1):
        qid = q['id']

        try:
            # 1. Retrieve
            results = hybrid_retrieve(q['question'])
            retrieved = get_chunks_by_results(results)

            # 2. Build prompt
            prompt = build_rag_prompt(q['question'], q['choices'], retrieved)

            # 3. Generate
            raw = ask_llm([
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': prompt},
            ], model=model)

            # 4. Parse
            pred = parse_answer(raw)
            if pred is None or pred < 1 or pred > 10:
                print(f"  ⚠️  Q{qid}: parse failed (raw='{str(raw)[:60]}'), defaulting to 9")
                pred = 9
                fallback_count += 1
            else:
                success_count += 1

            predictions[qid] = pred
            raw_responses[qid] = raw or ""

            # Progress display
            elapsed = time.time() - start_time
            done = i
            total_remaining = len(remaining)
            eta = elapsed / done * (total_remaining - done) if done < total_remaining else 0
            success_rate = success_count / i * 100
            print(f"  Q{qid:>3}: pred={pred:>2} | {q['question'][:38]:38s} | ETA {eta:.0f}s | ✓{success_rate:.0f}%")

            # FIX: Save checkpoint every 10 questions
            if i % 10 == 0:
                save_checkpoint(predictions, raw_responses)
                print(f"  💾 Checkpoint saved ({len(predictions)} predictions)")

        except Exception as e:
            print(f"  ❌ Q{qid}: Unexpected exception: {e}")
            predictions[qid] = 9
            raw_responses[qid] = str(e)
            fallback_count += 1

        if i < len(remaining):
            time.sleep(delay)

    # Final checkpoint save
    save_checkpoint(predictions, raw_responses)

    total_time = time.time() - start_time
    print(f"\n✅ Done! {len(predictions)} questions answered in {total_time:.0f}s")
    print(f"   Avg per question: {total_time/max(len(remaining),1):.1f}s")
    print(f"   Parse success: {success_count}/{len(remaining)} ({success_count/max(len(remaining),1)*100:.1f}%)")
    print(f"   Fallback to 9: {fallback_count}/{len(remaining)}")

    return predictions, raw_responses


#  RUN THE PIPELINE
predictions, raw_responses = run_rag_pipeline(questions, n=N_QUESTIONS)


🚀 Running RAG on 100 questions (model=openthaigpt)
   Already done: 0 | Remaining: 100
   Retrieval: Hybrid (Dense + BM25 RRF), top_k=10
   Delay between calls: 1.0s
  Q  1: pred= 5 | Watch S3 Ultra กันน้ำได้กี่ ATM ครับ   | ETA 573s | ✓100%
  Q  2: pred= 7 | ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ  | ETA 515s | ✓100%
  Q  3: pred= 2 | หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชั | ETA 542s | ✓100%
  ⚠️  502 server error, waiting 1.8s
  Q  4: pred= 6 | อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ | ETA 689s | ✓100%
  Q  5: pred= 6 | สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้ | ETA 630s | ✓100%
  Q  6: pred= 8 | ฟ้าใหม่จ่ายด้วย crypto ได้มั้ยคะ เช่น  | ETA 626s | ✓100%
  Q  7: pred= 1 | ซื้อ AirBook 14 มาแล้ว อยากรู้ว่าในกล่ | ETA 599s | ✓100%
  Q  8: pred= 9 | DuoPad สั่งซื้อได้เลยไหมครับ หรือต้องพ | ETA 582s | ✓100%
  Q  9: pred= 4 | เฮ้ยย อ่านเยอะมากเลย ขอถามนิดนึงนะคะ ค | ETA 627s | ✓100%
  Q 10: pred= 7 | อยากใช้ซิม 2 ค่ายพร้อมกันครับ ใส่ซิมได | ETA 657s | ✓100%
  💾 Checkpoint saved (10 predictions)

## Section 8: Eval Analysis

In [ ]:
# EVAL ANALYSIS

# 1. การกระจายตัวของตัวเลือกที่ระบบทำนาย (Distribution of predicted choices) - เช่น AI ตอบตัวเลือก 1, 2, 3, 4 ในสัดส่วนเท่าไหร่
# 2. คุณภาพของการค้นคืนข้อมูล (Retrieval quality) - วัดจากคะแนนความคล้ายคลึงของเวกเตอร์ (Cosine similarity scores)
# 3. การวิเคราะห์ความมั่นใจ (Confidence analysis) - แยกดูคำถามที่ระบบดึงข้อมูลมาได้คล้ายคลึงมาก (มั่นใจสูง) กับคล้ายคลึงน้อย (มั่นใจต่ำ)
# 4. สัดส่วนประเภทเอกสารที่ถูกดึงมาใช้ (Doc-type distribution of retrieved chunks) - ดูว่า AI ดึงข้อมูลมาจากเอกสารประเภทไหนบ้าง

from collections import Counter

print("📊 PREDICTION ANALYSIS")


# Choice distribution
choice_counts = Counter(predictions.values())
print("\n📈 Predicted choice distribution:")
for choice_num in range(1, 11):
    count = choice_counts.get(choice_num, 0)
    bar = '█' * count
    label = {
        9: '(ไม่มีข้อมูล)',
        10: '(ไม่เกี่ยวข้อง)'
    }.get(choice_num, '')
    print(f"  Choice {choice_num:>2}: {count:>3}x  {bar} {label}")

# 2. Retrieval quality analysis
print("\n🔍 RETRIEVAL QUALITY ANALYSIS")
print("="*50)

cosine_scores = []
for q in questions[:N_QUESTIONS]:
    d_results = dense_retrieve(q['question'], k=1)
    if d_results:
        cosine_scores.append(d_results[0][1])

if cosine_scores:
    print(f"  Top-1 cosine similarity stats:")
    print(f"    Mean:   {np.mean(cosine_scores):.4f}")
    print(f"    Median: {np.median(cosine_scores):.4f}")
    print(f"    Min:    {np.min(cosine_scores):.4f}")
    print(f"    Max:    {np.max(cosine_scores):.4f}")
    print(f"    Std:    {np.std(cosine_scores):.4f}")

    # Identify low-confidence questions (low cosine similarity)
    LOW_SIM_THRESHOLD = 0.5
    low_conf = [
        (questions[i]['id'], questions[i]['question'], cosine_scores[i])
        for i in range(len(cosine_scores))
        if cosine_scores[i] < LOW_SIM_THRESHOLD
    ]
    print(f"\n  ⚠️  Low confidence (sim < {LOW_SIM_THRESHOLD}): {len(low_conf)} questions")
    for qid, qtxt, score in sorted(low_conf, key=lambda x: x[2])[:10]:
        pred = predictions.get(qid, '?')
        print(f"    Q{qid:>3} [sim={score:.3f}, pred={pred}]: {qtxt[:60]}")

# 3. Doc-type retrieval analysis
print("\n📁 RETRIEVED DOC TYPE DISTRIBUTION")
print("="*50)
doc_type_counts = Counter()
for q in questions[:min(20, N_QUESTIONS)]:
    results = hybrid_retrieve(q['question'])
    for idx, _ in results:
        doc_type_counts[chunks[idx]['doc_type']] += 1

total_retrieved = sum(doc_type_counts.values())
for dt, count in doc_type_counts.most_common():
    pct = count / total_retrieved * 100
    print(f"  {dt:12s}: {count:>4} ({pct:.1f}%)")

# 4. Questions predicted as 9 or 10 (no data / irrelevant)
no_data_qs = [(qid, ans) for qid, ans in predictions.items() if ans >= 9]
print(f"\n📋 Questions answered as 9 or 10 (uncertain): {len(no_data_qs)}")
for qid, ans in no_data_qs:
    q_text = next(q['question'] for q in questions if q['id'] == qid)
    print(f"  Q{qid:>3} → choice {ans}: {q_text[:60]}")

📊 PREDICTION ANALYSIS

📈 Predicted choice distribution:
  Choice  1:  10x  ██████████ 
  Choice  2:  12x  ████████████ 
  Choice  3:   9x  █████████ 
  Choice  4:  16x  ████████████████ 
  Choice  5:  10x  ██████████ 
  Choice  6:  12x  ████████████ 
  Choice  7:  10x  ██████████ 
  Choice  8:   7x  ███████ 
  Choice  9:  14x  ██████████████ (ไม่มีข้อมูล)
  Choice 10:   0x   (ไม่เกี่ยวข้อง)

🔍 RETRIEVAL QUALITY ANALYSIS
  Top-1 cosine similarity stats:
    Mean:   0.6756
    Median: 0.6827
    Min:    0.3561
    Max:    0.8426
    Std:    0.0790

  ⚠️  Low confidence (sim < 0.5): 4 questions
    Q 63 [sim=0.356, pred=9]: สูตรผัดกระเพราหมูสับทำยังไงคะ?
    Q 60 [sim=0.383, pred=9]: วันหยุดราชการปี 2569 มีกี่วันครับ?
    Q 61 [sim=0.473, pred=9]: ตั๋วเครื่องบินกรุงเทพ-เชียงใหม่ราคาเท่าไหร่คะ?
    Q 62 [sim=0.484, pred=9]: ตอนนี้ดอกเบี้ยเงินฝากออมทรัพย์กี่เปอร์เซ็นต์คะ?

📁 RETRIEVED DOC TYPE DISTRIBUTION
  product     :  169 (84.5%)
  store_info  :   25 (12.5%)
  policy      :    6 (3.0%)

In [ ]:
# SELF-EVAL: Re-evaluate low-confidence questions

# สำหรับข้อคำถามที่โมเดล (AI) มีความไม่แน่ใจในคำตอบ เราสามารถแก้ปัญหาได้ดังนี้:
# 1. ดึงก้อนข้อมูลไปให้ AI อ่านมากขึ้น (เพิ่มค่า k ให้สูงขึ้น)
# 3. ลองใช้เทคนิคการเขียนคำถามใหม่ (Query rewriting) ให้ชัดเจนขึ้นก่อนไปค้นหา

def rerun_with_more_context(question_ids: list,
                              predictions: dict,
                              k_override: int = 10,
                              model_override: str = 'openthaigpt') -> dict:
    """Re-run specific questions with expanded retrieval."""
    updated = dict(predictions)  # Copy
    target = [q for q in questions if q['id'] in question_ids]

    print(f"🔄 Re-running {len(target)} questions with k={k_override}, model={model_override}")

    for q in target:
        qid = q['id']
        old_pred = predictions.get(qid)

        # More context retrieval
        results = hybrid_retrieve(q['question'], k=k_override)
        retrieved = get_chunks_by_results(results)
        prompt = build_rag_prompt(q['question'], q['choices'], retrieved)

        raw = ask_llm([
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ], model=model_override)

        pred = parse_answer(raw)
        if pred and 1 <= pred <= 10:
            updated[qid] = pred
            print(f"  Q{qid}: {old_pred} → {pred} ({q['question'][:40]})")

        time.sleep(0.5)

    return updated

print("ℹ️  Re-run cell ready. Uncomment lines above to use.")

ℹ️  Re-run cell ready. Uncomment lines above to use.


## Section 9: Generate Submission

In [ ]:
# Verify all 100 questions have predictions
all_ids = {q['id'] for q in questions}
pred_ids = set(predictions.keys())
missing = all_ids - pred_ids

if missing:
    print(f"⚠️  Missing predictions for: {sorted(missing)}")
    for mid in missing:
        predictions[mid] = 9  # Default to 'no data'
    print(f"   Defaulted to choice 9 for {len(missing)} questions")
else:
    print(f"✅ All {len(predictions)} questions have predictions")

# Validate all answers are in range 1-10
invalid = {qid: ans for qid, ans in predictions.items() if not (1 <= ans <= 10)}
if invalid:
    print(f"⚠️  Invalid answers: {invalid}")
    for qid in invalid:
        predictions[qid] = 9
else:
    print(f"✅ All answers are valid (1-10)")

# Write submission
write_submission(predictions, questions, OUTPUT_CSV)

# Preview
print(f"\n📋 Submission preview (first 10 rows):")
print(f"{'id':>4} | {'answer':>6} | question")
print("-" * 60)
for q in questions[:10]:
    qid = q['id']
    ans = predictions.get(qid, 9)
    choice_text = q['choices'].get(str(ans), '?')
    print(f"  {qid:>2} | {ans:>6} | {q['question'][:30]:30s} → {choice_text[:25]}")

✅ All 100 questions have predictions
✅ All answers are valid (1-10)
✅ Submission written to submission.csv (100 rows)

📋 Submission preview (first 10 rows):
  id | answer | question
------------------------------------------------------------
   1 |      5 | Watch S3 Ultra กันน้ำได้กี่ AT → 10 ATM
   2 |      7 | ปากกา SaiFah Pen Gen 2 ราคาเท่ → 3,990 บาท
   3 |      2 | หูฟัง HeadPro X1 ใช้ Bluetooth → Bluetooth 5.3
   4 |      6 | อยากเอามือถือเครื่องเก่ามาเทิร → ไม่มีบริการ Trade-in ในขณ
   5 |      6 | สั่งของจากฟ้าใหม่ สั่งได้ครั้ง → 10 รายการ
   6 |      8 | ฟ้าใหม่จ่ายด้วย crypto ได้มั้ย → ไม่รับชำระด้วย Cryptocurr
   7 |      1 | ซื้อ AirBook 14 มาแล้ว อยากรู้ → ตัวเครื่อง, หัวชาร์จ 65W,
   8 |      9 | DuoPad สั่งซื้อได้เลยไหมครับ ห → ไม่มีข้อมูลนี้ในฐานข้อมูล
   9 |      4 | เฮ้ยย อ่านเยอะมากเลย ขอถามนิดน → Rugged R1 กันน้ำระดับ IP6
  10 |      7 | อยากใช้ซิม 2 ค่ายพร้อมกันครับ  → X9 Pro ใส่ได้ 2 ซิมพร้อมก


In [ ]:
# Save detailed results for analysis
results_path = "detailed_results.json"

detailed = []
for q in questions:
    qid = q['id']
    ans = predictions.get(qid, 9)
    detailed.append({
        'id': qid,
        'question': q['question'],
        'predicted_choice': ans,
        'predicted_text': q['choices'].get(str(ans), ''),
        'llm_raw': raw_responses.get(qid, ''),
        'all_choices': q['choices'],
    })

with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(detailed, f, ensure_ascii=False, indent=2)

print(f"✅ Detailed results saved to {results_path}")

# Final summary
print(f"\n🏆 FINAL SUMMARY")
print(f"   Total questions: {len(predictions)}")
print(f"   Choice distribution:")
for c, cnt in sorted(Counter(predictions.values()).items()):
    print(f"     Choice {c}: {cnt}x")
print(f"\n📁 Output files:")
print(f"   {OUTPUT_CSV} — submit to Kaggle")
print(f"   {results_path} — detailed analysis")

✅ Detailed results saved to detailed_results.json

🏆 FINAL SUMMARY
   Total questions: 100
   Choice distribution:
     Choice 1: 10x
     Choice 2: 12x
     Choice 3: 9x
     Choice 4: 16x
     Choice 5: 10x
     Choice 6: 12x
     Choice 7: 10x
     Choice 8: 7x
     Choice 9: 14x

📁 Output files:
   submission.csv — submit to Kaggle
   detailed_results.json — detailed analysis
